# nb07 — TinyHuBERT Distillation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb07_distill.ipynb)

> **Shrink HuBERT 50× into a tiny, streamable featurizer you can run on a Raspberry Pi.**  \
> Train once on a GPU, then reuse the resulting `tinyhubert.onnx` as a CPU featurizer in
> `nb08_wakehubert.ipynb`.

---

## The idea: knowledge distillation

**Knowledge distillation** is training a small model (the **student**) to imitate the outputs
of a big model (the **teacher**). Instead of learning from scratch, the student learns to
reproduce what the teacher already knows — so it can reach surprisingly good quality at a tiny
fraction of the size.

Here:

- **Teacher** — frozen `facebook/hubert-base-ls960`. As explained in `nb06`, HuBERT is a
  self-supervised speech model that turns 16 kHz audio into rich 768-dimensional
  *frame embeddings* (~50 vectors per second). It is accurate but huge and slow.
- **Student** — `WakeHuBERTStudent`, a small **causal CNN + GRU** network. We train it so that,
  given the same audio, it outputs frame embeddings that match the teacher's.

```
Teacher (frozen HuBERT)   ~94 M params, ONNX ~350 MB, RTF 0.5–2.0 on CPU
   16 kHz waveform ─▶ [Transformer layers] ─▶ 768-d frame embeddings
                                                      ▲
                                          match these │ (MSE + InfoNCE loss)
                                                      ▼
Student (WakeHuBERTStudent)   ~1–4 M params, ONNX ~5–15 MB, RTF 0.005–0.02 on CPU
   16 kHz waveform ─▶ [causal CNN ×4] ─▶ [GRU] ─▶ 768-d frame embeddings
```

The student's CNN strides are sized so its total downsampling (×160 at 16 kHz) matches HuBERT's
frame rate — that lets us compare student and teacher frame-by-frame.

---

## Why "causal"?

The teacher is a Transformer: to embed any moment it looks at the *whole* clip, including the
future. An always-on wake word cannot wait for the future — it must decide as audio streams in.

The student is **causal** (also called *streaming*): each output frame depends only on past and
present audio, never future. A unidirectional GRU and left-padded ("causal") convolutions
guarantee this, so the exported model can process a sliding window in real time.

---

## The two-part loss

The student is trained to minimise `MSE + 0.1 × InfoNCE`:

- **MSE** (*mean squared error*) — the plain distance between the student's vector and the
  teacher's vector at each frame. It says "go to roughly this point in 768-d space". Easy to
  optimise, but on its own the student can cheat by collapsing every frame toward the teacher's
  average (low error, useless features).
- **InfoNCE** (*information noise-contrastive estimation*) — a **contrastive** loss. It lines up
  student frame *i* with teacher frame *i* (positive pair) and pushes it away from all other
  teacher frames in the batch (negatives). This forces *different* sounds to stay distinguishable
  and prevents the collapse that MSE alone allows.

MSE pulls toward the target; InfoNCE keeps the targets apart. The 0.1 weight lets MSE dominate
while InfoNCE acts as a guard rail.

---

## What this notebook does

| Cell | Step | What happens |
|------|------|-------------|
| 2 | **Configure** | Student dimensions, epochs, dataset, output path |
| 3 | **Install + GPU check** | Install deps; require CUDA for the teacher pass |
| 4 | **MLflow** | Optional experiment tracking |
| 5 | **Dataset** | Stream speech clips from HuggingFace, cache as arrays |
| 6 | **Distill** | Load teacher, build student, train with MSE + InfoNCE |
| 7 | **Visualise** | Loss curves + PCA overlap of student vs teacher embeddings |
| 8 | **Export ONNX** | Save and verify `tinyhubert.onnx` |
| 9 | **Downstream test** | Load it as a `ww_trainer` featurizer |
| 10 | **Summary** | Params, compression ratio, next steps |

**A GPU is required** (the teacher forward pass is impractical on CPU). On a Kaggle T4 a full
30-epoch run takes roughly 90 min; set `EPOCHS=10` for a quick smoke test. The student-only
ONNX, once exported, runs fine on CPU.

> **Relationship to the research script.** A richer, MLflow-driven command-line version of this
> recipe lives at `scripts/research/tinyhubert.py`, documented in
> `docs/research/tinyhubert.md`. This notebook is the self-contained, learn-by-reading variant:
> it imports `WakeHuBERTStudent` from that script when available and otherwise defines an
> equivalent student inline.

## Cell 2 — Configuration

Set the student's shape and the training options. Every value can also be supplied as an
environment variable of the same name.

Student architecture:

- `CNN_DIM` — base channel width of the convolutional front-end (bigger = more capacity).
- `GRU_HIDDEN` — hidden size of the recurrent layer.
- `OUTPUT_DIM` — must stay **768** to match HuBERT's embedding size, since the loss compares the
  two directly.

Training and data:

- `EPOCHS`, `BATCH_SIZE`, `LR` — standard training knobs.
- `MAX_AUDIO_SECS` — clips are truncated to this length.
- `DISTILL_DATASET` / `MAX_SAMPLES` — which HuggingFace speech dataset to stream and how many
  clips to collect. Any speech works; the student is learning HuBERT's representation, not the
  dataset's labels.
- `STUDENT_ONNX` — where the exported `tinyhubert.onnx` lands.
- `SKIP_DISTILL` — when true, reuse a cached dataset and an existing `tinyhubert.onnx` instead
  of redoing the expensive work. Set to `false` to force a fresh distillation run.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
DEVICE            = os.environ.get("DEVICE",            "cuda")
SEED              = int(os.environ.get("SEED",          "42"))

# ── Student architecture ──────────────────────────────────────────────────────
CNN_DIM           = int(os.environ.get("CNN_DIM",       "64"))
GRU_HIDDEN        = int(os.environ.get("GRU_HIDDEN",    "256"))
OUTPUT_DIM        = int(os.environ.get("OUTPUT_DIM",    "768"))  # must match HuBERT

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS            = int(os.environ.get("EPOCHS",        "30"))
BATCH_SIZE        = int(os.environ.get("BATCH_SIZE",    "32"))
LR                = float(os.environ.get("LR",          "3e-4"))
MAX_AUDIO_SECS    = float(os.environ.get("MAX_AUDIO_SECS", "5.0"))

# ── Dataset ───────────────────────────────────────────────────────────────────
LANG              = os.environ.get("LANG",              "en")
# HF dataset for distillation (streaming — no full download needed)
DISTILL_DATASET   = os.environ.get("DISTILL_DATASET",   "MLCommons/ml_spoken_words")
MAX_SAMPLES       = int(os.environ.get("MAX_SAMPLES",   "5000"))

# ── Output paths ─────────────────────────────────────────────────────────────
STUDENT_ONNX      = os.environ.get("STUDENT_ONNX",      f"{OUTPUT_DIR}/tinyhubert.onnx")
SKIP_DISTILL      = os.environ.get("SKIP_DISTILL",      "true").lower() == "true"

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "tinyhubert_distill")

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Student: CNN_DIM={CNN_DIM}, GRU_HIDDEN={GRU_HIDDEN}, OUTPUT_DIM={OUTPUT_DIM}")
print(f"Training: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, LR={LR}")
print(f"Output: {STUDENT_ONNX}")

## Cell 3 — Install dependencies and verify the GPU

Installs the scientific stack plus `transformers` / `accelerate` (to load the HuBERT teacher),
`datasets` (to stream training audio), and `ww_trainer`.

It then **requires a CUDA GPU**: distillation runs the full HuBERT teacher on every batch, which
is far too slow on CPU. If you already have a `tinyhubert.onnx`, you can skip straight to
`nb08_wakehubert.ipynb` and never need a GPU again.

> On Kaggle, select **GPU T4 × 1** under Session options → Accelerator first.

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "tqdm")
_pip("transformers", "accelerate", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. TinyHuBERT distillation requires GPU (teacher forward pass).\n"
        "If you already have tinyhubert.onnx, go directly to nb08_wakehubert.ipynb."
    )

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.1f} GB")
print(f"Platform: {_platform}")

## Cell 4 — MLflow tracking (optional)

**MLflow** is an experiment-tracking server that records metrics, parameters, and artifacts so
you can compare runs later. It is entirely optional — distillation works the same without it.

If `MLFLOW_URI` is set (and, on Kaggle, an `MLFLOW_TOKEN` secret exists), this cell points
MLflow at your server and names the experiment. Otherwise it prints that tracking is disabled
and moves on.

In [ ]:
import os

# ── MLflow setup ──────────────────────────────────────────────────────────────
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"No Kaggle secret: {e}")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    try:
        import mlflow
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        print(f"MLflow: {MLFLOW_EXPERIMENT!r} at {MLFLOW_URI}")
    except Exception as e:
        print(f"MLflow warning: {e}")
else:
    print("MLFLOW_URI not set — tracking disabled")

## Cell 5 — Collect the distillation audio

Distillation needs raw speech — the *labels do not matter*, since the student is only learning
to mimic the teacher's embeddings, not to classify anything. This cell **streams** clips from a
HuggingFace dataset (`MLCommons/ml_spoken_words` by default).

**Streaming** means clips are pulled one at a time over the network without downloading the
whole multi-terabyte dataset. Each clip is resampled to 16 kHz, truncated to `MAX_AUDIO_SECS`,
and very short clips (< 0.2 s) are skipped. Up to `MAX_SAMPLES` clips are collected into memory
and cached to a `.npz` file so re-runs are instant.

If the dataset cannot be reached (network or version issues), the cell falls back to a small
batch of random noise so the rest of the notebook still runs as a smoke test.

> Needs ~3 GB free disk for the cache.

In [ ]:
import numpy as np
import torch
from pathlib import Path

# ── Dataset: stream MLCommons/ml_spoken_words ─────────────────────────────────
# Streaming mode: no Arrow cache, no full download. Samples are fetched on demand.
# We collect up to MAX_SAMPLES audio clips as float32 arrays.

import shutil
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — need at least 3 GB."

_cache_file = Path(OUTPUT_DIR) / f"distill_wavs_{MAX_SAMPLES}.npz"

if SKIP_DISTILL and _cache_file.exists():
    print(f"Loading cached dataset from {_cache_file}...")
    data = np.load(str(_cache_file), allow_pickle=True)
    _all_wavs = list(data["wavs"])
    print(f"  Loaded {len(_all_wavs)} samples.")
else:
    print(f"Streaming {DISTILL_DATASET} ({LANG})...")
    from datasets import load_dataset
    import torchaudio

    try:
        ds = load_dataset(
            DISTILL_DATASET,
            LANG,
            split="train",
            streaming=True,
        )
    except Exception as e:
        print(f"Could not load {DISTILL_DATASET!r}: {e}")
        print("Falling back to random noise for a quick smoke-test.")
        _all_wavs = [np.random.randn(int(MAX_AUDIO_SECS * 16000)).astype(np.float32)
                     for _ in range(min(MAX_SAMPLES, 200))]
        ds = None

    if ds is not None:
        _all_wavs = []
        _audio_key = None
        for sample in ds:
            if _audio_key is None:
                _audio_key = "audio" if "audio" in sample else list(sample.keys())[0]
            audio_data = sample[_audio_key]
            if isinstance(audio_data, dict):
                wav_np = np.array(audio_data["array"], dtype=np.float32)
                sr = audio_data.get("sampling_rate", 16000)
            else:
                wav_np = np.array(audio_data, dtype=np.float32)
                sr = 16000
            if sr != 16000:
                wav_t = torch.from_numpy(wav_np).unsqueeze(0)
                wav_np = torchaudio.functional.resample(wav_t, sr, 16000).squeeze().numpy()
            max_len = int(MAX_AUDIO_SECS * 16000)
            wav_np = wav_np[:max_len]
            if len(wav_np) < 3200:  # skip very short clips (<0.2s)
                continue
            _all_wavs.append(wav_np.astype(np.float32))
            if len(_all_wavs) % 500 == 0:
                print(f"  Collected {len(_all_wavs)}/{MAX_SAMPLES} samples...")
            if len(_all_wavs) >= MAX_SAMPLES:
                break

    # Cache to disk
    import numpy.lib.format
    np.savez(_cache_file, wavs=np.array(_all_wavs, dtype=object))
    print(f"  Cached {len(_all_wavs)} samples → {_cache_file}")

print(f"Dataset: {len(_all_wavs)} samples, lengths {min(len(w) for w in _all_wavs)}–{max(len(w) for w in _all_wavs)} samples")

## Cell 6 — The distillation training loop

The heart of the notebook. Step by step:

1. **Get the student class.** It tries to import `WakeHuBERTStudent` from
   `scripts/research/tinyhubert.py`; if that file is not present, it defines an equivalent
   causal CNN + GRU student inline so the notebook is self-contained.
2. **Load the frozen teacher** — `facebook/hubert-base-ls960` plus its feature extractor, with
   all gradients turned off.
3. **Build the student** and an AdamW optimiser with a cosine-annealing learning-rate schedule.
4. **Train.** Each batch: run the teacher (no gradient) to get target 768-d frame embeddings,
   run the student to get its embeddings, trim both to the same number of frames, then compute
   `MSE + 0.1 × InfoNCE` and step the optimiser. Per-epoch MSE and InfoNCE are recorded.
5. **Save** the student checkpoint (`.pt`) and the loss history (`.json`).

When `SKIP_DISTILL=true` and `tinyhubert.onnx` already exists, the whole loop is skipped and the
saved history is loaded instead.

> This is the slow cell — expect the bulk of the ~90 min runtime here. Watch the printed MSE and
> InfoNCE fall epoch over epoch; both decreasing means the student is tracking the teacher.

In [ ]:
import sys
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# ── Distillation training ─────────────────────────────────────────────────────
# Strategy:
#   1. Try to import WakeHuBERTStudent from scripts/research/tinyhubert.py
#   2. Fall back to defining the student inline
#   3. Load frozen HuBERT teacher
#   4. Train student with MSE loss + InfoNCE contrastive loss
#
# The training loop saves a checkpoint per epoch.

STUDENT_ONNX_PATH = Path(STUDENT_ONNX)
STUDENT_PT_PATH   = STUDENT_ONNX_PATH.with_suffix(".pt")
TRAIN_CURVE_FILE  = Path(OUTPUT_DIR) / "distill_train_curve.json"

if SKIP_DISTILL and STUDENT_ONNX_PATH.exists():
    print(f"tinyhubert.onnx already exists at {STUDENT_ONNX_PATH} — skipping distillation.")
    print("Set SKIP_DISTILL=false to retrain.")
    _train_history = []
    if TRAIN_CURVE_FILE.exists():
        import json
        _train_history = json.loads(TRAIN_CURVE_FILE.read_text())
else:
    # ── Import or define student ───────────────────────────────────────────
    _tinyhubert_script = Path("scripts/research/tinyhubert.py")
    _student_imported = False
    if _tinyhubert_script.exists():
        try:
            import importlib.util
            spec = importlib.util.spec_from_file_location("tinyhubert", _tinyhubert_script)
            _th_mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(_th_mod)
            WakeHuBERTStudent = _th_mod.WakeHuBERTStudent
            _student_imported = True
            print(f"Imported WakeHuBERTStudent from {_tinyhubert_script}")
        except Exception as e:
            print(f"Could not import from tinyhubert.py: {e} — using inline definition")

    if not _student_imported:
        class CausalConv1d(nn.Module):
            """Causal 1D convolution (no future context)."""
            def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
                super().__init__()
                self.pad = (kernel_size - 1) * dilation
                self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                                      padding=0, dilation=dilation)
            def forward(self, x):
                x = torch.nn.functional.pad(x, (self.pad, 0))
                return self.conv(x)

        class WakeHuBERTStudent(nn.Module):
            """
            Streaming causal CNN + GRU student.
            Input : (B, T) float32 waveform at 16 kHz
            Output: (B, T', OUTPUT_DIM) frame embeddings — matches HuBERT shape
            """
            def __init__(self, cnn_dim=64, gru_hidden=256, output_dim=768, sr=16000):
                super().__init__()
                self.cnn = nn.Sequential(
                    CausalConv1d(1, cnn_dim, kernel_size=10, dilation=1),
                    nn.GELU(),
                    nn.GroupNorm(1, cnn_dim),
                    CausalConv1d(cnn_dim, cnn_dim, kernel_size=3, dilation=1),
                    nn.GELU(),
                    nn.GroupNorm(1, cnn_dim),
                    CausalConv1d(cnn_dim, cnn_dim * 2, kernel_size=3, dilation=2),
                    nn.GELU(),
                    nn.GroupNorm(1, cnn_dim * 2),
                )
                # After CNN, downsample ~20× to match HuBERT's 20ms frames
                self.pool = nn.AvgPool1d(kernel_size=20, stride=20)
                self.gru  = nn.GRU(cnn_dim * 2, gru_hidden, num_layers=2,
                                   batch_first=True, dropout=0.1)
                self.proj = nn.Linear(gru_hidden, output_dim)

            def forward(self, x):  # x: (B, T)
                h = x.unsqueeze(1)          # (B, 1, T)
                h = self.cnn(h)             # (B, C, T)
                h = self.pool(h)            # (B, C, T')
                h = h.transpose(1, 2)       # (B, T', C)
                h, _ = self.gru(h)          # (B, T', gru_hidden)
                return self.proj(h)         # (B, T', output_dim)

        print("Using inline WakeHuBERTStudent definition.")

    # ── Load teacher ───────────────────────────────────────────────────────
    from transformers import HubertModel, Wav2Vec2FeatureExtractor
    print("Loading HuBERT teacher (facebook/hubert-base-ls960)...")
    _teacher_processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
    _teacher = HubertModel.from_pretrained("facebook/hubert-base-ls960").to(DEVICE)
    _teacher.eval()
    for p in _teacher.parameters():
        p.requires_grad = False
    print(f"Teacher loaded. Params: {sum(p.numel() for p in _teacher.parameters()):,}")

    # ── Build student ──────────────────────────────────────────────────────
    student = WakeHuBERTStudent(
        cnn_dim=CNN_DIM, gru_hidden=GRU_HIDDEN, output_dim=OUTPUT_DIM
    ).to(DEVICE)
    n_params = sum(p.numel() for p in student.parameters())
    print(f"Student params: {n_params:,}  ({n_params/1e6:.2f}M)")

    optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    def _infonce_loss(student_out, teacher_out, temperature=0.07):
        """InfoNCE over frame embeddings: align student to teacher frames."""
        s = torch.nn.functional.normalize(student_out.reshape(-1, OUTPUT_DIM), dim=-1)
        t = torch.nn.functional.normalize(teacher_out.reshape(-1, OUTPUT_DIM), dim=-1)
        n = s.shape[0]
        logits = torch.mm(s, t.T) / temperature  # (N, N)
        labels = torch.arange(n, device=DEVICE)
        return torch.nn.functional.cross_entropy(logits, labels)

    _train_history = []
    import random
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print(f"\nStarting distillation: {EPOCHS} epochs, {len(_all_wavs)} samples...")
    for epoch in range(1, EPOCHS + 1):
        student.train()
        indices = list(range(len(_all_wavs)))
        random.shuffle(indices)
        ep_mse, ep_nce, ep_n = 0.0, 0.0, 0

        for batch_start in range(0, len(indices), BATCH_SIZE):
            batch_idx = indices[batch_start : batch_start + BATCH_SIZE]
            # Pad to same length within batch
            batch_wavs = [_all_wavs[i] for i in batch_idx]
            max_len = max(len(w) for w in batch_wavs)
            padded = np.zeros((len(batch_wavs), max_len), dtype=np.float32)
            for i, w in enumerate(batch_wavs):
                padded[i, :len(w)] = w
            wav_t = torch.from_numpy(padded).to(DEVICE)  # (B, T)

            with torch.no_grad():
                # Teacher: process through HuBERT feature extractor + encoder
                inputs = _teacher_processor(
                    [w.tolist() for w in padded],
                    sampling_rate=16000,
                    return_tensors="pt",
                    padding=True,
                ).input_values.to(DEVICE)
                teacher_out = _teacher(inputs).last_hidden_state  # (B, T', 768)

            student_out = student(wav_t)  # (B, T'', OUTPUT_DIM)

            # Align temporal dimensions (student may differ slightly)
            min_t = min(student_out.shape[1], teacher_out.shape[1])
            s_aligned = student_out[:, :min_t, :]
            t_aligned = teacher_out[:, :min_t, :]

            mse = torch.nn.functional.mse_loss(s_aligned, t_aligned)
            nce = _infonce_loss(s_aligned[:16], t_aligned[:16])  # limit for speed
            loss = mse + 0.1 * nce

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()

            ep_mse += mse.item()
            ep_nce += nce.item()
            ep_n += 1

        scheduler.step()
        ep_mse /= ep_n
        ep_nce /= ep_n
        _train_history.append({"epoch": epoch, "mse": ep_mse, "nce": ep_nce})
        print(f"  Epoch {epoch:3d}/{EPOCHS}  MSE={ep_mse:.4f}  InfoNCE={ep_nce:.4f}")

    # Save checkpoint
    torch.save(student.state_dict(), str(STUDENT_PT_PATH))
    print(f"Checkpoint saved: {STUDENT_PT_PATH}")

    # Save training curve
    import json
    TRAIN_CURVE_FILE.write_text(json.dumps(_train_history, indent=2))
    print(f"Train curve saved: {TRAIN_CURVE_FILE}")

## Cell 7 — Visualise the distillation

Two diagnostics:

1. **Loss curves** — MSE and InfoNCE plotted against epoch. Both should trend downward; a flat
   or rising curve means training stalled (try a lower `LR` or more data).
2. **PCA overlap** — reloads the trained student, embeds ~50 clips with both student and
   teacher, and projects them to 2-D with **PCA** (*principal component analysis*, a linear
   dimensionality reducer). If the blue (student) and red (teacher) point clouds **overlap**,
   the student has learned to live in the same representation space as the teacher — the goal of
   distillation. Well-separated clouds mean the student has not converged.

Both plots are saved as PNGs under `OUTPUT_DIR`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# ── Training curve ────────────────────────────────────────────────────────────
if _train_history:
    epochs = [r["epoch"] for r in _train_history]
    mse_vals = [r["mse"] for r in _train_history]
    nce_vals = [r["nce"] for r in _train_history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("TinyHuBERT distillation training curve", fontsize=12)
    axes[0].plot(epochs, mse_vals, color="steelblue", linewidth=1.5)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE loss")
    axes[0].set_title("Frame-level MSE (student vs teacher)")
    axes[1].plot(epochs, nce_vals, color="coral", linewidth=1.5)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("InfoNCE loss")
    axes[1].set_title("InfoNCE contrastive loss")
    plt.tight_layout()
    curve_png = str(Path(OUTPUT_DIR) / "distill_train_curve.png")
    plt.savefig(curve_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Training curve saved: {curve_png}")
    print(f"Final: MSE={mse_vals[-1]:.4f}  InfoNCE={nce_vals[-1]:.4f}")
else:
    print("No training history (distillation was skipped).")

# ── PCA overlap: student vs teacher ───────────────────────────────────────────
# Sample a few hundred frames from the validation wavs.
# Project both student and teacher embeddings to 2D.
# Good overlap = student has learned the teacher's representation space.

if not STUDENT_ONNX_PATH.exists() and not STUDENT_PT_PATH.exists():
    print("No student model yet — skipping PCA.")
else:
    import torch
    from sklearn.decomposition import PCA

    print("Running PCA overlap analysis...")
    try:
        if not _student_imported:
            pass  # student class defined above in this session
        _eval_student = WakeHuBERTStudent(
            cnn_dim=CNN_DIM, gru_hidden=GRU_HIDDEN, output_dim=OUTPUT_DIM
        ).to(DEVICE)
        if STUDENT_PT_PATH.exists():
            _eval_student.load_state_dict(torch.load(str(STUDENT_PT_PATH), map_location=DEVICE))
        _eval_student.eval()

        # Take first 50 samples
        _pca_wavs = _all_wavs[:50]
        _s_embs, _t_embs = [], []
        with torch.no_grad():
            for wav in _pca_wavs:
                wav_t = torch.from_numpy(wav[np.newaxis, :]).to(DEVICE)
                s_out = _eval_student(wav_t).mean(dim=1).cpu().numpy()  # (1, 768)
                _s_embs.append(s_out[0])
            inputs = _teacher_processor(
                [w.tolist() for w in _pca_wavs],
                sampling_rate=16000, return_tensors="pt", padding=True
            ).input_values.to(DEVICE)
            t_out = _teacher(inputs).last_hidden_state.mean(dim=1).cpu().numpy()  # (N, 768)
            _t_embs = list(t_out)

        all_embs = np.vstack(_s_embs + _t_embs)
        labels_st = ["student"] * len(_s_embs) + ["teacher"] * len(_t_embs)
        pca = PCA(n_components=2, random_state=SEED)
        coords = pca.fit_transform(all_embs)
        n = len(_s_embs)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(coords[:n, 0], coords[:n, 1], c="steelblue", s=20, alpha=0.7, label="student")
        ax.scatter(coords[n:, 0], coords[n:, 1], c="coral", s=20, alpha=0.7, marker="^", label="teacher")
        ax.set_title("PCA overlap: student (blue) vs teacher (red)\nGood distillation → clusters overlap")
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        ax.legend(fontsize=9)
        plt.tight_layout()
        pca_path = str(Path(OUTPUT_DIR) / "distill_pca_overlap.png")
        plt.savefig(pca_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"PCA overlap plot saved: {pca_path}")
    except Exception as e:
        print(f"PCA analysis failed: {e}")

## Cell 8 — Export and verify the ONNX featurizer

Exports the trained student to `tinyhubert.onnx` (skipped if the file already exists), with a
dynamic time axis so it accepts audio of any length. It then reloads the file with
`onnxruntime` and runs a dummy waveform through it to confirm the output shape is
`(batch, frames, 768)` — proof the file is valid and ready to use without PyTorch.

In [ ]:
import torch
import numpy as np
from pathlib import Path

# ── ONNX export ───────────────────────────────────────────────────────────────

STUDENT_ONNX_PATH = Path(STUDENT_ONNX)

if STUDENT_ONNX_PATH.exists():
    print(f"tinyhubert.onnx already exists: {STUDENT_ONNX_PATH}")
    print(f"  Size: {STUDENT_ONNX_PATH.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("Exporting student to ONNX...")
    _eval_student.eval()
    dummy_input = torch.randn(1, 16000).to(DEVICE)
    torch.onnx.export(
        _eval_student,
        dummy_input,
        str(STUDENT_ONNX_PATH),
        input_names=["waveform"],
        output_names=["embeddings"],
        dynamic_axes={
            "waveform":   {0: "batch", 1: "time"},
            "embeddings": {0: "batch", 1: "frames"},
        },
        opset_version=17,
        do_constant_folding=True,
    )
    size_mb = STUDENT_ONNX_PATH.stat().st_size / 1024 / 1024
    print(f"ONNX exported: {STUDENT_ONNX_PATH}  ({size_mb:.1f} MB)")

# Verify with onnxruntime
import onnxruntime as ort
sess = ort.InferenceSession(str(STUDENT_ONNX_PATH), providers=["CPUExecutionProvider"])
dummy_np = np.random.randn(1, 16000).astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy_np})
print(f"ONNX verification OK: output shape={out[0].shape}  (expected (1, T', 768))")

## Cell 9 — Use it as a `ww_trainer` featurizer

Confirms the end goal: loading `tinyhubert.onnx` through `ww_trainer.feats.OnnxFeatureExtractor`
— the same interface the embedded featurizers use. A dummy clip is passed through to print the
output shape, the feature vector that a wake-word classifier head will consume in
`nb08_wakehubert.ipynb`. If `ww_trainer` is not importable, it falls back to raw `onnxruntime`.

In [ ]:
import numpy as np
from pathlib import Path

# ── Quick downstream test: use tinyhubert.onnx as OnnxFeatureExtractor ────────

try:
    from ww_trainer.feats import OnnxFeatureExtractor
    extractor = OnnxFeatureExtractor(str(STUDENT_ONNX_PATH))
    dummy_wav = np.random.randn(16000).astype(np.float32)
    feat = extractor(dummy_wav)
    print(f"OnnxFeatureExtractor output shape: {feat.shape}")
    print("→ This is the embedding that gets fed to the wake-word classifier (nb08).")
except ImportError:
    # Manual verification via onnxruntime
    import onnxruntime as ort
    sess = ort.InferenceSession(str(STUDENT_ONNX_PATH), providers=["CPUExecutionProvider"])
    dummy_wav = np.random.randn(1, 16000).astype(np.float32)
    out = sess.run(None, {sess.get_inputs()[0].name: dummy_wav})
    print(f"onnxruntime output shape: {out[0].shape}")
    print("ww_trainer.feats.OnnxFeatureExtractor not found — using raw onnxruntime.")

print()
print(f"tinyhubert.onnx is ready for use in nb08_wakehubert.ipynb.")
print(f"Path: {STUDENT_ONNX_PATH}")

## Cell 10 — Summary and next steps

Prints the teacher vs student parameter counts, the **compression ratio** (how many times
smaller the student is), the ONNX size, and the final losses. It ends with the one thing you
need for the next stage: the path to `tinyhubert.onnx` to plug into `nb08_wakehubert.ipynb` as
`FEATURIZER_ONNX`, where you train an actual wake-word classifier on top of this distilled
featurizer.

In [ ]:
from pathlib import Path
import json

# ── Summary ───────────────────────────────────────────────────────────────────

STUDENT_ONNX_PATH = Path(STUDENT_ONNX)

print("=" * 60)
print("TinyHuBERT distillation summary")
print("=" * 60)

# Teacher params
try:
    t_params = sum(p.numel() for p in _teacher.parameters())
    print(f"Teacher (HuBERT-base) : {t_params:,} params  (~{t_params/1e6:.0f}M)")
except NameError:
    print("Teacher              : facebook/hubert-base-ls960  (~94M params)")

# Student params
try:
    s_params = sum(p.numel() for p in _eval_student.parameters())
    print(f"Student (TinyHuBERT) : {s_params:,} params  ({s_params/1e6:.2f}M)")
    print(f"Compression ratio    : {t_params / s_params:.0f}×")
except NameError:
    print("Student              : WakeHuBERTStudent (see above)")

if STUDENT_ONNX_PATH.exists():
    print(f"ONNX size            : {STUDENT_ONNX_PATH.stat().st_size / 1024 / 1024:.1f} MB")

if _train_history:
    final = _train_history[-1]
    print(f"Final MSE            : {final['mse']:.4f}")
    print(f"Final InfoNCE        : {final['nce']:.4f}")

print()
print("Next step: use tinyhubert.onnx as the featurizer in nb08_wakehubert.ipynb")
print(f"  FEATURIZER_ONNX={STUDENT_ONNX_PATH}")
print("=" * 60)